In [ ]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated
import os

In [ ]:
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [3]:
class JokeState(TypedDict):
    topic: str
    joke: Annotated[list[BaseMessage], add_messages]
    explaination: Annotated[list[BaseMessage], add_messages]

In [4]:
def generate_joke(state: JokeState):
    response = model.invoke(f"generate shayari on the following topic \n {state['topic']}").content
    return {'joke' : response}

In [ ]:
def generate_explanation(state: JokeState):
    response = model.invoke(f"generate explanation of the following joke \n {state['joke']}").content
    return {'explaination' : response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable" : {'thread_id' : '1'}}

workflow.invoke({'topic' : 'any english shayari'}, config=config1)

{'topic': 'any english shayari',
 'joke': [HumanMessage(content='**दिल‑टूटने की शायरी – कुछ मशहूर शायरों के शब्दों में**\n\n---\n\n### १. मिर्ज़ा ग़ालिब  \n> **हज़ारों ख़्वाहिशें ऐसी कि हर ख़्वाहिश पे दम निकले,**  \n> **बहुत निकले मेरे अरमान लेकिन फिर भी कम निकले।**  \n\n*ग़ालिब की इस ग़ज़ल में अधूरी आशाओं और दिल‑टूटने की पीड़ा को बयां किया गया है।*  \n\n---\n\n### २. फ़ैज़ अहमद फ़ैज़  \n> **राहों में बिखरे थे ख़्वाबों के टुकड़े,**  \n> **अब तो बस यादों की धुंध में खोया हूँ मैं।**  \n\n*फ़ैज़ की इस पंक्तियों में वह दर्द है, जब प्यार के रास्ते पर बिखरे ख़्वाब अब सिर्फ़ यादों की धुंध बन कर रह गए।*  \n\n---\n\n### ३. गुलज़ार  \n> **जाने वाले को अलविदा कहूँ तो क्या,**  \n> **दिल में बसी है एक धुंधली सी याद,**  \n> **जो हर साँस में फिर से तड़पती है।**  \n\n*गुलज़ार की इस शायरी में बिछड़ने के बाद भी दिल में बसी अनकही ख़ुशी‑दर्द की झलक मिलती है।*  \n\n---\n\n### ४. जावेद अख़्तर  \n> **दिल तोड़ने वाले को क्या पता,**  \n> **दिल की धड़कन कितनी नाज़ुक होती है,**  \n> **एक झटके में बिखर जाता है सा

In [12]:
print(workflow.get_state(config1).values)

{'topic': 'any english shayari', 'joke': [HumanMessage(content='**दिल‑टूटने की शायरी – कुछ मशहूर शायरों के शब्दों में**\n\n---\n\n### १. मिर्ज़ा ग़ालिब  \n> **हज़ारों ख़्वाहिशें ऐसी कि हर ख़्वाहिश पे दम निकले,**  \n> **बहुत निकले मेरे अरमान लेकिन फिर भी कम निकले।**  \n\n*ग़ालिब की इस ग़ज़ल में अधूरी आशाओं और दिल‑टूटने की पीड़ा को बयां किया गया है।*  \n\n---\n\n### २. फ़ैज़ अहमद फ़ैज़  \n> **राहों में बिखरे थे ख़्वाबों के टुकड़े,**  \n> **अब तो बस यादों की धुंध में खोया हूँ मैं।**  \n\n*फ़ैज़ की इस पंक्तियों में वह दर्द है, जब प्यार के रास्ते पर बिखरे ख़्वाब अब सिर्फ़ यादों की धुंध बन कर रह गए।*  \n\n---\n\n### ३. गुलज़ार  \n> **जाने वाले को अलविदा कहूँ तो क्या,**  \n> **दिल में बसी है एक धुंधली सी याद,**  \n> **जो हर साँस में फिर से तड़पती है।**  \n\n*गुलज़ार की इस शायरी में बिछड़ने के बाद भी दिल में बसी अनकही ख़ुशी‑दर्द की झलक मिलती है।*  \n\n---\n\n### ४. जावेद अख़्तर  \n> **दिल तोड़ने वाले को क्या पता,**  \n> **दिल की धड़कन कितनी नाज़ुक होती है,**  \n> **एक झटके में बिखर जाता है सार

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'heartbreak in hindi, by famous shayar', 'joke': [HumanMessage(content='**दिल‑टूटने की शायरी (ग़ालिब‑शैली में)**  \n\n```\nदिल‑टूटे तो क्या, ज़ख्मों की दवा नहीं मिलती,\nहर ख़ुशी के बाद, बस एक ख़ालीपन की स्याही बिखरती।\n\nवो लफ़्ज़ जो कभी मोहब्बत के थे, अब धुंध में खो गए,\nसिर्फ़ यादों की परछाइयाँ, दिल के कोने‑कोने में रोते।\n\nइश्क़ की राह में हर कदम पर, आँसू‑आँसू बिखरते,\nजैसे बरसात में बूँदें, फिर भी धूप की आस नहीं मिलती।\n\nकहते हैं ग़ालिब, “हज़ारों ख़्वाहिशें ऐसी कि हर ख़्वाहिश पे दम निकले”,\nपर अब तो बस एक ही ख़्वाहिश बची‑‑कि फिर से दिल धड़कना शुरू हो जाए।\n```\n\n*— इस शायरी को मिर्ज़ा ग़ालिब की शैली से प्रेरित होकर लिखा गया है।*', additional_kwargs={}, response_metadata={}, id='f9cd3893-ab18-4b55-b31e-06a5c87baf68')], 'explaination': [HumanMessage(content='## शायरी का संक्षिप्त सार  \n\nयह शायरी दिल‑टूटने, अकेलेपन और फिर से दिल\u202fको\u202fधड़कने\u202fकी\u202fइच्छा\u202fपर\u202fकेन्द्रित\u202fहै।  \nग़ालिब‑शैली की झलक‑झलक शब्द‑बारी, ग़ज़ल‑के‑सिलस

In [31]:
config2 = {'configurable' : {'thread_id' : '2'}}
workflow.invoke({'topic' : 'enthusiasm ke uppar'}, config=config2)

{'topic': 'enthusiasm ke uppar',
 'joke': [HumanMessage(content='**उत्साह के ऊपर एक शायरी**\n\nउत्साह की राहों में जब कदम बढ़ते हैं,  \nहर ख़्वाब को पंख मिलते हैं, उड़ते हैं।  \nजो दिल में जलता है वो शिख़ा नहीं,  \nसिर्फ़ एक चिंगारी, जो आग बन जाती है।  \n\nहौसले की धूप में बूँदें नहीं गिरतीं,  \nहर गिरते‑गिरते भी मुस्कुराहट बनती।  \nजो थक कर बैठता है, वो नहीं देख पाता,  \nउत्साह की लहरों में हर मोड़ पर नया सवेरा आता।  \n\nतो चलो, इस जोश को दिल में बिठा लो,  \nहर मंज़िल को अपनी राह बना लो।  \nउत्साह के ऊपर, सपनों की छाँव में,  \nजिंदगी को फिर से रंगीन बना लो।', additional_kwargs={}, response_metadata={}, id='cb281347-a0ad-41a9-980c-3e7e683bc070')],
 'explaination': [HumanMessage(content='**शायरी का सार‑संग्रह (Explanation)**  \n\n| शायरी की पंक्तियाँ | अर्थ (सामान्य भाषा में) | प्रमुख काव्यात्मक तत्व / छवि |\n|-------------------|--------------------------|------------------------------|\n| **उत्साह की राहों में जब कदम बढ़ते हैं,** | जब हम जोश‑जुनून से आगे बढ़ते हैं, तो… | *रूपक* – “उत्

In [32]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'enthusiasm ke uppar', 'joke': [HumanMessage(content='**उत्साह के ऊपर एक शायरी**\n\nउत्साह की राहों में जब कदम बढ़ते हैं,  \nहर ख़्वाब को पंख मिलते हैं, उड़ते हैं।  \nजो दिल में जलता है वो शिख़ा नहीं,  \nसिर्फ़ एक चिंगारी, जो आग बन जाती है।  \n\nहौसले की धूप में बूँदें नहीं गिरतीं,  \nहर गिरते‑गिरते भी मुस्कुराहट बनती।  \nजो थक कर बैठता है, वो नहीं देख पाता,  \nउत्साह की लहरों में हर मोड़ पर नया सवेरा आता।  \n\nतो चलो, इस जोश को दिल में बिठा लो,  \nहर मंज़िल को अपनी राह बना लो।  \nउत्साह के ऊपर, सपनों की छाँव में,  \nजिंदगी को फिर से रंगीन बना लो।', additional_kwargs={}, response_metadata={}, id='cb281347-a0ad-41a9-980c-3e7e683bc070')], 'explaination': [HumanMessage(content='**शायरी का सार‑संग्रह (Explanation)**  \n\n| शायरी की पंक्तियाँ | अर्थ (सामान्य भाषा में) | प्रमुख काव्यात्मक तत्व / छवि |\n|-------------------|--------------------------|------------------------------|\n| **उत्साह की राहों में जब कदम बढ़ते हैं,** | जब हम जोश‑जुनून से आगे बढ़ते हैं,

In [39]:
workflow.invoke({'topic' : 'same as previous chat topic but from different shayar and in urdu language only'}, config=config1)

{'topic': 'same as previous chat topic but from different shayar and in urdu language only',
 'joke': [HumanMessage(content='**दिल‑टूटने की शायरी (ग़ालिब‑शैली में)**  \n\n```\nदिल‑टूटे तो क्या, ज़ख्मों की दवा नहीं मिलती,\nहर ख़ुशी के बाद, बस एक ख़ालीपन की स्याही बिखरती।\n\nवो लफ़्ज़ जो कभी मोहब्बत के थे, अब धुंध में खो गए,\nसिर्फ़ यादों की परछाइयाँ, दिल के कोने‑कोने में रोते।\n\nइश्क़ की राह में हर कदम पर, आँसू‑आँसू बिखरते,\nजैसे बरसात में बूँदें, फिर भी धूप की आस नहीं मिलती।\n\nकहते हैं ग़ालिब, “हज़ारों ख़्वाहिशें ऐसी कि हर ख़्वाहिश पे दम निकले”,\nपर अब तो बस एक ही ख़्वाहिश बची‑‑कि फिर से दिल धड़कना शुरू हो जाए।\n```\n\n*— इस शायरी को मिर्ज़ा ग़ालिब की शैली से प्रेरित होकर लिखा गया है।*', additional_kwargs={}, response_metadata={}, id='f9cd3893-ab18-4b55-b31e-06a5c87baf68'),
  HumanMessage(content='Sure! Could you let me know which topic you’d like the shayari to be about? Once I have the theme, I’ll craft a beautiful piece for you.', additional_kwargs={}, response_metadata={}, id='39

In [40]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'same as previous chat topic but from different shayar and in urdu language only', 'joke': [HumanMessage(content='**दिल‑टूटने की शायरी (ग़ालिब‑शैली में)**  \n\n```\nदिल‑टूटे तो क्या, ज़ख्मों की दवा नहीं मिलती,\nहर ख़ुशी के बाद, बस एक ख़ालीपन की स्याही बिखरती।\n\nवो लफ़्ज़ जो कभी मोहब्बत के थे, अब धुंध में खो गए,\nसिर्फ़ यादों की परछाइयाँ, दिल के कोने‑कोने में रोते।\n\nइश्क़ की राह में हर कदम पर, आँसू‑आँसू बिखरते,\nजैसे बरसात में बूँदें, फिर भी धूप की आस नहीं मिलती।\n\nकहते हैं ग़ालिब, “हज़ारों ख़्वाहिशें ऐसी कि हर ख़्वाहिश पे दम निकले”,\nपर अब तो बस एक ही ख़्वाहिश बची‑‑कि फिर से दिल धड़कना शुरू हो जाए।\n```\n\n*— इस शायरी को मिर्ज़ा ग़ालिब की शैली से प्रेरित होकर लिखा गया है।*', additional_kwargs={}, response_metadata={}, id='f9cd3893-ab18-4b55-b31e-06a5c87baf68'), HumanMessage(content='Sure! Could you let me know which topic you’d like the shayari to be about? Once I have the theme, I’ll craft a beautiful piece for you.', additional_kwargs={}, response_